In [1]:
!pip install pandas numpy matplotlib seaborn tqdm -q


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import warnings
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 60)
sns.set_theme(style='whitegrid', font_scale=1.05)

PALETTE = {
    'CPSC':       '#4C72B0',
    'CPSC_Extra': '#DD8452',
    'INCART':     '#55A868',
    'Georgia':    '#C44E52',
    'PTB_XL':     '#8172B2',
}

In [3]:
# Папка с датасетами — поменяй на свой путь
BASE = Path('data/physionet2020')

DATASETS = {
    'CPSC':       BASE / 'Training_WFDB',
    'CPSC_Extra': BASE / 'Training_2',
    'INCART':     BASE / 'Training_StPetersburg',
    'Georgia':    BASE / 'WFDB_GEORGIA',
    'PTB_XL':     BASE / 'WFDB_PTB-XL',
    'MIT-BIH':    BASE / 'mit-bih',
}

N_WORKERS = 5

In [4]:
SNOMED = {
    '426783006': 'Normal sinus rhythm',
    '164889003': 'Atrial fibrillation',
    '164890007': 'Atrial flutter',
    '426627000': 'Bradycardia',
    '427084000': 'Sinus tachycardia',
    '427393003': 'Sinus bradycardia',
    '426177001': 'Sinus bradycardia',
    '713422000': 'Sinus arrhythmia',
    '17338001':  'Premature ventricular complex',
    '164884008': 'Ventricular ectopics',
    '427172004': 'Premature atrial complex',
    '284470004': 'Premature atrial complex',
    '63593006':  'Supraventricular premature beats',
    '251168009': 'Supraventricular tachycardia',
    '270492004': '1st degree AV block',
    '195042002': '2nd degree AV block',
    '27885002':  '3rd degree AV block',
    '713426002': 'Incomplete RBBB',
    '713427006': 'Complete RBBB',
    '164909002': 'LBBB',
    '445118002': 'Left anterior fascicular block',
    '251146004': 'Low QRS voltage',
    '164931005': 'ST depression',
    '164930006': 'ST elevation',
    '164934002': 'T-wave abnormality',
    '59931005':  'T-wave inversion',
    '429622005': 'ST depression',
    '111975006': 'Prolonged QT',
    '77867006':  'Short QT',
    '55827005':  'Left ventricular hypertrophy',
    '89792004':  'Right ventricular hypertrophy',
    '446358003': 'Right axis deviation',
    '47665007':  'Right axis deviation',
    '251200008': 'Left axis deviation',
    '67751000':  'Left axis deviation',
    '164865005': 'Myocardial infarction',
    '413444003': 'Acute MI',
    '59118001':  'RBBB',
    '428750005': 'Non-specific intraventricular block',
    '233917008': 'AV junctional rhythm',
    '10370003':  'Pacing rhythm',
    '164947007': 'Prolonged PR',
    '365413008': 'Pacemaker rhythm',
    '698252002': 'Non-specific ST-T change',
    '426761007': 'Supraventricular tachycardia',
    '164896001': 'Ventricular flutter',
    '111288001': 'Ventricular tachycardia',
    '195060002': 'Ventricular premature beats',
    '39732003': 'ECG: left axis deviation',
    '164873001': 'ECG: left ventricle hypertrophy',
    '164951009': 'QRS complex abnormal',
    '164861001': 'ECG: myocardial ischaemia',
    '67741000119109': 'Left atrial enlargement',
    '427393009': 'ECG: sinus arrhythmia',
    '164867002': 'ECG: old myocardial infarction',
    '425623009': 'ECG: lateral ischaemia',
    '164917005': 'Q wave abnormal',
    '55930002': 'ST segment changes',
    '425419005': 'ECG: inferior ischaemia',
    '54329005': 'Acute myocardial infarction of anterior wall',
    '426434006': 'ECG: anterior ischaemia',
    '251120003': 'Incomplete left bundle branch block',
    '445211001': 'ECG: left posterior fascicular block',
    '413844008': 'Chronic myocardial ischaemia',
    '428417006': 'Early repolarization',
    '6374002': 'Bundle branch block',
    '266249003': 'Ventricular hypertrophy',
    '11157007': 'Ventricular bigeminy',
    '74390002': 'Wolff-Parkinson-White pattern',
    '253352002': 'Left atrial abnormality',
    '195126007': 'Atrial hypertrophy',
    '251268003': 'ECG: atrial pacing pattern',
    '251266004': 'ECG: ventricular pacing pattern',
    '195080001': 'Atrial fibrillation and flutter',
    '446813000': 'Left atrial hypertrophy',
    '251180001': 'Ventricular trigeminy',
    '67198005': 'Paroxysmal supraventricular tachycardia',
    '251182009': 'Paired ventricular premature complexes',
    '426664006': 'ECG: accelerated junctional rhythm',
    '53741008': 'Coronary arteriosclerosis',
    '425856008': 'ECG: paroxysmal ventricular tachycardia',
    '253339007': 'Right atrial abnormality',
    '251139008': 'Suspect arm ECG leads reversed',
    '164921003': 'R wave abnormal',
    '426995002': 'ECG: junctional escape rhythm',
    '65778007': 'Sinoatrial block',
    '266257000': 'Transient ischaemic attack',
    '13640000': 'Fusion beats',
    '195101003': 'ECG: wandering atrial pacemaker',
    '426648003': 'ECG: junctional tachycardia',
    '29320008': 'Ectopic rhythm',
    '57054005': 'Acute myocardial infarction',
    '251170000': 'Blocked premature atrial contraction',
    '49578007': 'Shortened PR interval',
    '75532003': 'Ventricular escape beat',
    '54016002': 'Mobitz type I incomplete atrioventricular block',
    '251173003': 'Atrial bigeminy',
    '251164006': 'Junctional premature complex',
    '74615001': 'Tachycardia-bradycardia',
    '164895002': 'ECG: ventricular tachycardia',
    '81898007': 'Ventricular escape rhythm',
    '60423000': 'Sinus node dysfunction',
    '49260003': 'Idioventricular rhythm',
    '370365005': 'ECG: left ventricular strain',
    '704997005': 'Inferior ST segment depression',
    '82226007': 'Diffuse intraventricular block',
    '251259000': 'High T-voltage',
    '164937009': 'U wave abnormal',
    '426749004': 'Chronic atrial fibrillation',
    '282825002': 'Paroxysmal atrial fibrillation',
}

In [5]:
STD_KEYS    = frozenset({'age', 'sex', 'dx', 'rx', 'hx', 'sx'})
_RE_COMMENT = re.compile(r'#([^:]+):\s*(.*)')
_NAN_VALS   = frozenset(('nan', 'unknown', '', 'n/a', 'null'))


def parse_hea(path: str) -> dict:
    rec = {
        'record_id': Path(path).stem, 'filepath': path,
        'num_leads': None, 'fs': None, 'num_samples': None, 'duration_s': None,
        'age': None, 'sex': None, 'dx_codes': [], 'dx_names': [],
    }
    try:
        with open(path, 'r', errors='replace', buffering=2048) as fh:
            raw = fh.read(4096)
    except OSError:
        return rec

    lines = raw.splitlines()
    if lines:
        p = lines[0].split()
        try:
            rec['num_leads']   = int(p[1])
            rec['fs']          = float(p[2])
            rec['num_samples'] = int(p[3])
            rec['duration_s']  = round(rec['num_samples'] / rec['fs'], 2)
        except (IndexError, ValueError, ZeroDivisionError):
            pass

    for line in lines:
        if not line or line[0] != '#':
            continue
        m = _RE_COMMENT.match(line)
        if not m:
            continue
        k_raw, v = m.group(1).strip(), m.group(2).strip()
        k = k_raw.lower()

        if k == 'age':
            if v.lower() not in _NAN_VALS:
                try: rec['age'] = float(v)
                except ValueError: pass
        elif k == 'sex':
            if v.lower() not in _NAN_VALS: rec['sex'] = v
        elif k == 'dx':
            codes = [c.strip() for c in v.split(',') if c.strip()]
            rec['dx_codes'] = codes
            rec['dx_names'] = [SNOMED.get(c, f'Unknown ({c})') for c in codes]

    return rec


def collect_hea(path) -> list:
    out = []
    for root, _, files in os.walk(path):
        for fn in files:
            if fn.endswith('.hea'):
                out.append(os.path.join(root, fn))
    return out


def load_dataset(name: str, path, workers: int = N_WORKERS) -> pd.DataFrame:
    files = collect_hea(path)
    rows  = [None] * len(files)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futs = {pool.submit(parse_hea, f): i for i, f in enumerate(files)}
        for fut in tqdm(as_completed(futs), total=len(files), desc=name, leave=False):
            rows[futs[fut]] = fut.result()
    df = pd.DataFrame(rows)
    df['dataset'] = name
    return df

In [6]:
dfs = {
    name: load_dataset(name, path)
    for name, path in DATASETS.items()
    if Path(path).is_dir()
}
df = pd.concat(dfs.values(), ignore_index=True)

CPSC:   0%|          | 0/6877 [00:00<?, ?it/s]

CPSC_Extra:   0%|          | 0/3453 [00:00<?, ?it/s]

INCART:   0%|          | 0/74 [00:00<?, ?it/s]

Georgia:   0%|          | 0/10344 [00:00<?, ?it/s]

PTB_XL:   0%|          | 0/21837 [00:00<?, ?it/s]

MIT-BIH: 0it [00:00, ?it/s]

In [8]:
from collections import Counter

unknown_counter = Counter()
unknown_examples = {}

for _, row in df.iterrows():
    for code in row['dx_codes']:
        if code not in SNOMED:
            unknown_counter[code] += 1
            unknown_examples.setdefault(code, []).append(row['record_id'])

unknown_df = pd.DataFrame([
    {'code': code, 'count': cnt, 'examples': unknown_examples[code][:3]}
    for code, cnt in unknown_counter.most_common()
])

print(f'Unique unknown codes: {len(unknown_df)}')
print(f'Total unknown code occurrences: {unknown_counter.total() if unknown_counter else 0}')
unknown_df

Unique unknown codes: 0
Total unknown code occurrences: 0


""


In [27]:
df[~df['dx_names'].str.contains('Normal sinus rhythm', na=False)]

,record_id,filepath,num_leads,fs,num_samples,duration_s,age,sex,dx_codes,dx_names,dataset
0,A0001,data\physionet2020\Training_WFDB\A0001.hea,12.0,500.0,7500.0,15.00,74.0,Male,[59118001],[RBBB],CPSC
1,A0002,data\physionet2020\Training_WFDB\A0002.hea,12.0,500.0,5000.0,10.00,49.0,Female,[426783006],[Normal sinus rhythm],CPSC
2,A0003,data\physionet2020\Training_WFDB\A0003.hea,12.0,500.0,5000.0,10.00,81.0,Female,[164889003],[Atrial fibrillation],CPSC
3,A0004,data\physionet2020\Training_WFDB\A0004.hea,12.0,500.0,5974.0,11.95,45.0,Male,[164889003],[Atrial fibrillation],CPSC
4,A0005,data\physionet2020\Training_WFDB\A0005.hea,12.0,500.0,12500.0,25.00,53.0,Male,[164884008],[Ventricular ectopics],CPSC
...,...,...,...,...,...,...,...,...,...,...,...
42580,HR21833,data\physionet2020\WFDB_PTB-XL\HR21833.hea,12.0,500.0,5000.0,10.00,67.0,Female,"[164873001, 164884008, 164934002, 39732003, 427084000]","[ECG: left ventricle hypertrophy, Ventricular ectopics, ...",PTB_XL
42581,HR21834,data\physionet2020\WFDB_PTB-XL\HR21834.hea,12.0,500.0,5000.0,10.00,93.0,Male,"[164951009, 426783006]","[QRS complex abnormal, Normal sinus rhythm]",PTB_XL
42582,HR21835,data\physionet2020\WFDB_PTB-XL\HR21835.hea,12.0,500.0,5000.0,10.00,59.0,Female,"[164861001, 426783006]","[ECG: myocardial ischaemia, Normal sinus rhythm]",PTB_XL
42583,HR21836,data\physionet2020\WFDB_PTB-XL\HR21836.hea,12.0,500.0,5000.0,10.00,64.0,Female,"[39732003, 426783006]","[ECG: left axis deviation, Normal sinus rhythm]",PTB_XL
